# 02 — Sampling, Randomization & Confounders

Between forming a hypothesis and collecting data sits a critical design question:
**how do you draw your sample and assign users to groups?**

Getting this wrong contaminates every downstream analysis — no statistical test
can fix a broken sampling or assignment procedure.

This notebook covers:

1. **Simple random sampling** — the baseline and its limitations
2. **What is a confounder?** — variables that corrupt causal inference
3. **Why randomization works** — the mechanism that makes A/B testing valid
4. **Stratified randomization** — guarantee balance on known confounders at design time
5. **Hash-based assignment** — how companies implement randomization at scale
6. **Covariate balance check** — verify randomization actually worked post-assignment
7. **Regression adjustment (ANCOVA)** — correct for imbalance if balance fails
8. **Bootstrapping** — CIs on any metric without distributional assumptions
9. **Multi-armed bandits** — adaptive sampling as an alternative to fixed A/B

In [ ]:
import sys
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sys.path.append("../src")
from ab_stats import two_proportion_ztest

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
np.random.seed(42)

df = pd.read_csv("../data/ab_data_clean.csv")
print(f"Loaded: {len(df):,} rows")
print(df[["group","converted","age","device_type","gender","session_duration"]].head())

## 1. Simple Random Sampling

The baseline: assign each user to control or treatment with equal probability,
independently of any other attribute.

**The problem:** with finite samples, pure random assignment can produce unlucky
imbalances. With 1,000 users, you might get 540 mobile users in treatment but only
380 in control. This imbalance inflates variance and makes segment analysis unreliable.

More critically: if mobile users convert at a different rate, this imbalance can
make the treatment look better or worse than it actually is. That is a confounder.

In [ ]:
# Demonstrate imbalance under simple random assignment
n_sims = 1_000
n_users = 1_000
mobile_frac = 0.60

imbalances = []
for _ in range(n_sims):
    devices = np.random.choice(["mobile", "desktop"], size=n_users,
                               p=[mobile_frac, 1 - mobile_frac])
    treatment = np.random.binomial(1, 0.5, n_users)
    mobile_trt  = (devices == "mobile")[treatment == 1].mean()
    mobile_ctrl = (devices == "mobile")[treatment == 0].mean()
    imbalances.append(abs(mobile_trt - mobile_ctrl))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(imbalances, bins=40, color="steelblue", alpha=0.8, edgecolor="white")
p95 = np.percentile(imbalances, 95)
ax.axvline(p95, color="tomato", linestyle="--",
           label="95th pct: {:.1%} imbalance".format(p95))
ax.set_xlabel("Absolute difference in mobile fraction (treatment vs control)")
ax.set_ylabel("Frequency")
ax.set_title("Simple random: mobile imbalance over {:,} simulations (N={})".format(n_sims, n_users))
ax.legend()
plt.tight_layout()
plt.show()

print("Mean imbalance:  {:.2%}".format(np.mean(imbalances)))
print("95th percentile: {:.2%}".format(np.percentile(imbalances, 95)))
print("Max imbalance:   {:.2%}".format(np.max(imbalances)))

## 2. What Is a Confounder?

A **confounder** is a variable that is correlated with both:
- The **treatment assignment** (who ends up in control vs treatment), AND
- The **outcome** (the metric you are measuring)

When a confounder is present, you cannot tell whether the treatment caused the
change in the outcome, or whether the confounder is responsible.

### Concrete example

Suppose you launch a new landing page on your mobile app first (before rolling it
out to desktop). Mobile users are disproportionately in the treatment group.

Mobile users also convert at a lower rate than desktop users.

Result: the treatment group has more mobile users so treatment looks worse than it
actually is. The *true* causal effect of the page design is hidden.

```
              CONFOUNDER: Device Type
                   /              \
                  /                \
    Treatment Assignment ------>  Conversion Rate
```

The arrow from device type to treatment is what creates the problem.
In a proper randomized experiment, that arrow is severed by design.

In [ ]:
# Simulate confounding: observational study vs randomized experiment
rng = np.random.default_rng(42)
N = 20_000
TRUE_LIFT = 0.05   # new page truly improves conversion by +5pp for everyone

# Confounder: device type (mobile converts less)
device = rng.choice(["mobile", "desktop"], size=N, p=[0.60, 0.40])
base_rate = np.where(device == "mobile", 0.08, 0.20)

# Observational: new page rolled out on mobile app first
# Mobile users have 70% chance of seeing new page; desktop only 30%
p_trt = np.where(device == "mobile", 0.70, 0.30)
trt_obs = rng.binomial(1, p_trt)
conv_obs = rng.binomial(1, np.clip(base_rate + trt_obs * TRUE_LIFT, 0, 1))
obs_lift = conv_obs[trt_obs == 1].mean() - conv_obs[trt_obs == 0].mean()

# Randomized: 50/50 assignment independent of device
trt_rand = rng.binomial(1, 0.5, N)
conv_rand = rng.binomial(1, np.clip(base_rate + trt_rand * TRUE_LIFT, 0, 1))
rand_lift = conv_rand[trt_rand == 1].mean() - conv_rand[trt_rand == 0].mean()

mobile_trt_obs  = (device[trt_obs == 1] == "mobile").mean()
mobile_ctrl_obs = (device[trt_obs == 0] == "mobile").mean()
mobile_trt_rand  = (device[trt_rand == 1] == "mobile").mean()
mobile_ctrl_rand = (device[trt_rand == 0] == "mobile").mean()

print("=== Observational study (confounded) ===")
print("Mobile in treatment: {:.1%}   Mobile in control: {:.1%}".format(
      mobile_trt_obs, mobile_ctrl_obs))
print("Naive lift estimate: {:+.4f}".format(obs_lift))
print("True lift:           {:+.4f}".format(TRUE_LIFT))
print("Bias:                {:+.4f}".format(obs_lift - TRUE_LIFT))
print()
print("=== Randomized experiment (unconfounded) ===")
print("Mobile in treatment: {:.1%}   Mobile in control: {:.1%}".format(
      mobile_trt_rand, mobile_ctrl_rand))
print("Estimated lift:      {:+.4f}".format(rand_lift))
print("True lift:           {:+.4f}".format(TRUE_LIFT))
print("Bias:                {:+.4f}".format(rand_lift - TRUE_LIFT))

In [ ]:
# Visualise: confounding bias vs unbiased randomized estimate
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
groups = ["Control", "Treatment"]
obs_mob  = [mobile_ctrl_obs,  mobile_trt_obs]
rand_mob = [mobile_ctrl_rand, mobile_trt_rand]
x = np.arange(2)
w = 0.35
ax.bar(x - w/2, [v * 100 for v in obs_mob],  width=w, color="tomato",
       label="Observational", alpha=0.8)
ax.bar(x + w/2, [v * 100 for v in rand_mob], width=w, color="steelblue",
       label="Randomized", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(groups)
ax.set_ylabel("Mobile users (%)")
ax.set_title("Device mix: observational vs randomized")
ax.axhline(60, color="gray", linestyle=":", linewidth=1, label="True population (60%)")
ax.legend(fontsize=9)

ax = axes[1]
estimates = [obs_lift, rand_lift, TRUE_LIFT]
labels    = ["Observational
(confounded)", "Randomized
(unbiased)", "True lift"]
colors    = ["tomato", "steelblue", "gray"]
bars = ax.bar(labels, [v * 100 for v in estimates], color=colors, alpha=0.8)
ax.set_ylabel("Estimated lift (percentage points)")
ax.set_title("Confounding biases the estimate; randomization fixes it")
for bar, val in zip(bars, estimates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            "{:+.2f}pp".format(val * 100), ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

## 3. Why Randomization Eliminates Confounders

Randomization works because it **severs the link between the confounder and treatment
assignment**. When assignment is random:

- Mobile users are equally likely to be in control or treatment (50/50)
- Desktop users are equally likely to be in control or treatment (50/50)
- Any other user characteristic — observed or *unobserved* — is similarly balanced

This is the fundamental mechanism of A/B testing. In the language of causal
inference: randomization makes the **potential outcomes** independent of treatment.

$$Y(0), Y(1) \perp T \quad \text{(randomization assumption)}$$

A naive comparison of means in the treatment and control groups then gives an
unbiased estimate of the **average treatment effect (ATE)**.

### Observed vs unobserved confounders

| Method | Handles observed confounders | Handles unobserved confounders |
|---|---|---|
| Randomized A/B test | Yes (by design) | **Yes** (key advantage) |
| Observational methods | If measured and modeled | No |

Propensity score matching, DiD, and RDD can only adjust for confounders you
measured. A/B testing makes the question irrelevant by design.

## 4. Stratified Randomization

Even with full randomization, small samples can produce unlucky imbalances on
known covariates. **Stratified randomization** forces balance by pre-defining
strata and assigning independently within each one.

**How it works:** within "Mobile" users, assign exactly 50% to treatment.
Within "Desktop" users, assign exactly 50% to treatment.
The overall split is exactly 50/50 on that dimension by construction.

**Benefits:**
- Eliminates unlucky imbalances on dimensions you care about
- Reduces variance of the treatment effect estimate (same mechanism as CUPED)
- Segment analysis is cleaner — no compositional bias within strata

**Limitation:** you can only stratify on variables you know at assignment time.
Randomization still handles unobserved confounders — stratification just tightens
balance on the ones you name.

In [ ]:
def stratified_assign(df, strata_col, treatment_frac=0.5, seed=42):
    """Assign treatment within each stratum independently."""
    rng = np.random.default_rng(seed)
    assigned = []
    for stratum, group in df.groupby(strata_col):
        n = len(group)
        n_trt = int(round(n * treatment_frac))
        labels = np.array(["control"] * (n - n_trt) + ["treatment"] * n_trt)
        rng.shuffle(labels)
        assigned.append(pd.Series(labels, index=group.index))
    return pd.concat(assigned)

# Compare: simple vs stratified on device_type
n_sims = 1_000
simple_imb, strat_imb = [], []
sample = df[["device_type", "converted"]].copy()

for seed in range(n_sims):
    rng = np.random.default_rng(seed)
    simple_assign = rng.choice(["control", "treatment"], size=len(sample))
    m_trt  = (sample["device_type"] == "Mobile")[simple_assign == "treatment"].mean()
    m_ctrl = (sample["device_type"] == "Mobile")[simple_assign == "control"].mean()
    simple_imb.append(abs(m_trt - m_ctrl))

    strat_assign = stratified_assign(sample, "device_type", seed=seed)
    m_trt_s  = (sample["device_type"] == "Mobile")[strat_assign == "treatment"].mean()
    m_ctrl_s = (sample["device_type"] == "Mobile")[strat_assign == "control"].mean()
    strat_imb.append(abs(m_trt_s - m_ctrl_s))

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(simple_imb, bins=40, alpha=0.6, color="tomato",    label="Simple random")
ax.hist(strat_imb,  bins=40, alpha=0.6, color="steelblue", label="Stratified")
ax.set_xlabel("Mobile fraction imbalance (|treatment - control|)")
ax.set_ylabel("Frequency")
ax.set_title("Stratified randomization eliminates device imbalance")
ax.legend()
plt.tight_layout()
plt.show()

print("Simple random  - mean imbalance: {:.4%}  max: {:.4%}".format(
      np.mean(simple_imb), np.max(simple_imb)))
print("Stratified     - mean imbalance: {:.4%}  max: {:.4%}".format(
      np.mean(strat_imb), np.max(strat_imb)))

## 5. Hash-Based Assignment

In a live system you cannot pre-generate a list of assignments — users arrive
continuously and you need an instant, consistent, database-free answer to
"which group is this user in?"

**The solution:** hash the user ID + experiment ID to a bucket number.

```python
bucket = int(hashlib.md5(f"{user_id}_{experiment_id}".encode()).hexdigest(), 16) % 100
group  = "treatment" if bucket < 50 else "control"
```

**Properties:**
- **Deterministic:** the same user always lands in the same bucket for the same experiment
- **Independent across experiments:** adding the experiment ID means a user in treatment for
  experiment A is essentially randomly assigned for experiment B
- **No database lookup:** instant assignment at request time
- **Auditable:** you can reproduce any assignment at any time

**The experiment ID is critical.** Without it, users who are in treatment for
experiment A will tend to be in treatment for B, C, D — creating spurious
correlations between concurrent experiments.

In [ ]:
def hash_assign(user_id, experiment_id, treatment_pct=50):
    """Deterministic hash-based group assignment."""
    key = "{}_{}".format(user_id, experiment_id).encode()
    bucket = int(hashlib.md5(key).hexdigest(), 16) % 100
    return "treatment" if bucket < treatment_pct else "control"

n_users = 100_000
user_ids = range(n_users)

assign_exp1 = [hash_assign(u, "exp_landing_page_v1")  for u in user_ids]
assign_exp2 = [hash_assign(u, "exp_checkout_flow_v3") for u in user_ids]

exp1_trt_rate = assign_exp1.count("treatment") / n_users
exp2_trt_rate = assign_exp2.count("treatment") / n_users

exp1_bin = np.array([1 if a == "treatment" else 0 for a in assign_exp1])
exp2_bin = np.array([1 if a == "treatment" else 0 for a in assign_exp2])
corr = np.corrcoef(exp1_bin, exp2_bin)[0, 1]

print("Experiment 1 treatment rate: {:.4f}  (target: 0.50)".format(exp1_trt_rate))
print("Experiment 2 treatment rate: {:.4f}  (target: 0.50)".format(exp2_trt_rate))
print("Correlation between exp1 and exp2 assignments: {:.4f}  (target: ~0)".format(corr))
print()
print("Same user, different experiment IDs:")
for uid in [42, 1337, 99999, 7]:
    g1 = hash_assign(uid, "exp_landing_page_v1")
    g2 = hash_assign(uid, "exp_checkout_flow_v3")
    print("  user {:<6}: exp1={:<12} exp2={}".format(uid, g1, g2))

## 6. Covariate Balance Check

After assigning users — whether via simple random, stratified, or hash — you
should **verify** that key covariates are balanced across groups.

This is the A/B testing equivalent of "Table 1" in a clinical trial paper.

### Standardized Mean Difference (SMD)

For each covariate:

$$\text{SMD} = \frac{\bar{x}_{\text{treatment}} - \bar{x}_{\text{control}}}{s_{\text{pooled}}}$$

**Rule of thumb:** |SMD| < 0.10 indicates good balance.
|SMD| > 0.20 warrants investigation; > 0.50 means the covariate is strongly imbalanced.

For categorical variables, compute SMD for each level as a binary (0/1) indicator.

**A good balance check catches:**
- Unlucky random imbalance on high-impact dimensions
- Bugs in the assignment logic that non-randomly routed users
- Selection bias if the experiment started mid-funnel

In [ ]:
ctrl = df[df["group"] == "control"]
trt  = df[df["group"] == "treatment"]

def smd(x_ctrl, x_trt):
    """Standardized mean difference."""
    pooled_sd = np.sqrt((np.var(x_ctrl, ddof=1) + np.var(x_trt, ddof=1)) / 2)
    return 0.0 if pooled_sd == 0 else (x_trt.mean() - x_ctrl.mean()) / pooled_sd

balance = []
for col in ["age", "session_duration", "pages_visited"]:
    d = smd(ctrl[col].values, trt[col].values)
    balance.append({"covariate": col,
                    "control_mean": ctrl[col].mean(),
                    "treatment_mean": trt[col].mean(),
                    "smd": d})

for col in ["device_type", "gender"]:
    for val in sorted(df[col].unique()):
        x_c = (ctrl[col] == val).astype(float).values
        x_t = (trt[col]  == val).astype(float).values
        balance.append({"covariate": "{}={}".format(col, val),
                        "control_mean": x_c.mean(),
                        "treatment_mean": x_t.mean(),
                        "smd": smd(x_c, x_t)})

bal = pd.DataFrame(balance)
bal["|smd|"] = bal["smd"].abs()
bal["balanced"] = bal["|smd|"] < 0.10

print("Covariate Balance Table")
print("=" * 68)
print("{:<25} {:>12} {:>12} {:>8} {:>8}".format(
      "Covariate", "Control", "Treatment", "SMD", "OK?"))
print("-" * 68)
for _, row in bal.iterrows():
    flag = "OK" if row["balanced"] else "CHECK"
    print("{:<25} {:>12.4f} {:>12.4f} {:>8.4f} {:>8}".format(
          row["covariate"], row["control_mean"],
          row["treatment_mean"], row["smd"], flag))

In [ ]:
# Love plot: visualise all SMDs at a glance
fig, ax = plt.subplots(figsize=(8, len(bal) * 0.45 + 1))
colors = ["tomato" if v > 0.10 else "steelblue" for v in bal["|smd|"]]
ax.barh(bal["covariate"], bal["|smd|"], color=colors, alpha=0.8)
ax.axvline(0.10, color="tomato", linestyle="--", linewidth=1.2,
           label="|SMD| = 0.10 threshold")
ax.set_xlabel("Absolute Standardized Mean Difference")
ax.set_title("Love Plot: Covariate Balance
|SMD| < 0.10 = good balance")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

n_bad = (~bal["balanced"]).sum()
if n_bad == 0:
    print("All {} covariates balanced (|SMD| < 0.10). Randomization worked.".format(len(bal)))
else:
    print("{} of {} covariates have |SMD| >= 0.10:".format(n_bad, len(bal)))
    print(bal[~bal["balanced"]][["covariate","smd"]].to_string(index=False))

## 7. Regression Adjustment (ANCOVA)

In a well-randomized experiment, ANCOVA does NOT fix confounding — randomization
already did that. What it does:

1. **Reduces residual variance** — by explaining variation through covariates,
   the error term shrinks → smaller SE → better statistical power
2. **Corrects post-hoc imbalance** — if a covariate is imbalanced despite
   randomization (bad luck with small N), including it adjusts the estimate

**The model:**

$$Y_i = \beta_0 + \beta_1 \cdot \text{treatment}_i + \beta_2 \cdot X_i + \varepsilon_i$$

The coefficient $\beta_1$ gives the treatment effect holding covariates constant.

> **Connection to CUPED (notebook 09):** CUPED is ANCOVA applied to pre-experiment
> covariates. The math is identical — subtract the covariate-predicted portion of Y.
> ANCOVA is the general form; CUPED is the industry shorthand for this technique.

In [ ]:
# ANCOVA: adjust treatment effect for age and session_duration using OLS
y             = df["converted"].values.astype(float)
trt_indicator = (df["group"] == "treatment").astype(float).values
age_c         = (df["age"] - df["age"].mean()).values
sess_c        = (df["session_duration"] - df["session_duration"].mean()).values

# Design matrix: intercept, treatment, age (centered), session_duration (centered)
X_mat = np.column_stack([np.ones(len(df)), trt_indicator, age_c, sess_c])

# OLS: beta = (X^T X)^{-1} X^T y
beta, _, _, _ = np.linalg.lstsq(X_mat, y, rcond=None)

# SE from residual variance
resid   = y - X_mat @ beta
n_obs, n_par = X_mat.shape
s2      = (resid @ resid) / (n_obs - n_par)
cov_b   = s2 * np.linalg.inv(X_mat.T @ X_mat)
se_trt  = np.sqrt(cov_b[1, 1])
z_anc   = beta[1] / se_trt
p_anc   = 2 * stats.norm.sf(abs(z_anc))

# Unadjusted estimate
ctrl_rate = df[df["group"] == "control"]["converted"].mean()
trt_rate  = df[df["group"] == "treatment"]["converted"].mean()
unadj = trt_rate - ctrl_rate

print("Unadjusted treatment effect:  {:+.6f}".format(unadj))
print()
print("ANCOVA-adjusted (age + session_duration):")
print("  Lift: {:+.6f}".format(beta[1]))
print("  SE:   {:.6f}".format(se_trt))
print("  z:    {:.3f}".format(z_anc))
print("  p:    {:.4f}".format(p_anc))
print()
print("In a well-randomized experiment the two estimates should be near-identical.")
print("Any small difference reflects residual covariate imbalance that ANCOVA corrects.")

## 8. Bootstrapping

The z-test and t-test assume your metric is approximately normally distributed.
For conversion rates with large N this is fine — the CLT applies.

But many real metrics are highly skewed:
- Revenue per user (most users spend $0, a few spend thousands)
- Session duration (heavy right tail)
- 90th percentile load time
- Median order value

For these, **bootstrapping** gives valid confidence intervals without any
distributional assumption. Repeatedly resample your data with replacement,
compute the metric on each resample, and use the distribution of resampled
estimates as a proxy for sampling variability.

**Algorithm:**
1. Draw N samples with replacement from your data
2. Compute the metric on this bootstrap sample
3. Repeat B times (typically 1,000–10,000)
4. The 2.5th and 97.5th percentiles of bootstrap estimates = 95% CI

In [ ]:
ctrl_rev = df[df["group"] == "control"]["purchase_amount"].values
trt_rev  = df[df["group"] == "treatment"]["purchase_amount"].values
B = 5_000

def bootstrap_ci(data, stat_fn, B=5_000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    estimates = [stat_fn(rng.choice(data, size=len(data), replace=True))
                 for _ in range(B)]
    lo = np.percentile(estimates, 100 * alpha / 2)
    hi = np.percentile(estimates, 100 * (1 - alpha / 2))
    return np.array(estimates), lo, hi

# Bootstrap CI on mean revenue difference
rng = np.random.default_rng(99)
diff_boots = [
    rng.choice(trt_rev,  len(trt_rev),  replace=True).mean() -
    rng.choice(ctrl_rev, len(ctrl_rev), replace=True).mean()
    for _ in range(B)
]
diff_lo = np.percentile(diff_boots, 2.5)
diff_hi = np.percentile(diff_boots, 97.5)

print("Bootstrap 95% CI on mean revenue per user:")
print("  Control:    ${:.4f}".format(ctrl_rev.mean()))
print("  Treatment:  ${:.4f}".format(trt_rev.mean()))
print("  Difference: ${:+.4f}  CI: [${:+.4f}, ${:+.4f}]".format(
      trt_rev.mean() - ctrl_rev.mean(), diff_lo, diff_hi))
print("  CI contains zero: {}".format(diff_lo < 0 < diff_hi))

# Bootstrap CI on median difference (no parametric equivalent exists)
rng2 = np.random.default_rng(77)
med_boots = [
    np.median(rng2.choice(trt_rev,  len(trt_rev),  replace=True)) -
    np.median(rng2.choice(ctrl_rev, len(ctrl_rev), replace=True))
    for _ in range(B)
]
med_lo = np.percentile(med_boots, 2.5)
med_hi = np.percentile(med_boots, 97.5)
print()
print("Bootstrap 95% CI on MEDIAN revenue difference:")
print("  [{:+.4f}, {:+.4f}]".format(med_lo, med_hi))
print("  (No parametric equivalent -- bootstrap is the right tool here)")

In [ ]:
# Visualise: skewed revenue distribution and bootstrap sampling distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(ctrl_rev[ctrl_rev > 0], bins=60, color="steelblue", alpha=0.6,
        label="Control (converters only)", density=True)
ax.hist(trt_rev[trt_rev > 0],  bins=60, color="coral",     alpha=0.6,
        label="Treatment (converters only)", density=True)
ax.set_xlabel("Purchase amount ($)")
ax.set_ylabel("Density")
ax.set_title("Revenue is right-skewed -- bootstrap CI is safer than z-test")
ax.legend(fontsize=9)

ax = axes[1]
ax.hist(diff_boots, bins=60, color="steelblue", alpha=0.8, density=True)
ax.axvline(0,       color="gray",   linewidth=1.5, linestyle="--", label="No difference")
ci_label = "95% CI: [{:+.3f}, {:+.3f}]".format(diff_lo, diff_hi)
ax.axvline(diff_lo, color="tomato", linewidth=1.5, linestyle=":", label=ci_label)
ax.axvline(diff_hi, color="tomato", linewidth=1.5, linestyle=":")
ax.set_xlabel("Bootstrapped difference in mean revenue ($/user)")
ax.set_title("Bootstrap sampling distribution of the treatment effect")
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 9. Multi-Armed Bandits

Standard A/B testing has a fixed exploration phase (run the experiment) followed
by a fixed exploitation phase (ship the winner). This is efficient when the cost
of running the experiment is low.

**When A/B is expensive:** if users in the losing variant lose revenue or churn,
every user sent to the loser during the experiment is a cost. Bandits reduce this
cost by adaptively shifting traffic toward the better variant as evidence accumulates.

### Thompson Sampling
Maintain a Beta(α, β) posterior for each arm's conversion rate. At each
step, sample from each arm posterior and route the user to the arm with the
highest sample. Naturally balances exploration and exploitation.

**The tradeoff vs fixed A/B:**

| | Fixed A/B | Multi-Armed Bandit |
|---|---|---|
| Statistical guarantees | Strong (controlled α, power) | Weaker (regret bounds) |
| User cost during experiment | High (50% on loser) | Low (shifts to winner) |
| Detects subtle effects | Yes (with enough N) | Often misses small effects |
| Best for | Confirmatory decisions | Fast iteration, high user cost |
| Used by | Most product teams | Personalisation, content ranking |

In [ ]:
TRUE_RATE_A = 0.10
TRUE_RATE_B = 0.14
N_ROUNDS    = 10_000

# Thompson Sampling
alpha_a, beta_a = 1, 1
alpha_b, beta_b = 1, 1
ts_regret, ts_traffic_b = [], []

rng = np.random.default_rng(42)
for t in range(1, N_ROUNDS + 1):
    sample_a = rng.beta(alpha_a, beta_a)
    sample_b = rng.beta(alpha_b, beta_b)
    if sample_b > sample_a:
        reward = rng.binomial(1, TRUE_RATE_B)
        alpha_b += reward; beta_b += (1 - reward)
        ts_traffic_b.append(1)
    else:
        reward = rng.binomial(1, TRUE_RATE_A)
        alpha_a += reward; beta_a += (1 - reward)
        ts_traffic_b.append(0)
    ts_regret.append(TRUE_RATE_B - (TRUE_RATE_B if ts_traffic_b[-1] else TRUE_RATE_A))

# Fixed 50/50
ab_regret = []
for t in range(N_ROUNDS):
    chosen = rng.choice([0, 1])
    ab_regret.append(TRUE_RATE_B - (TRUE_RATE_B if chosen else TRUE_RATE_A))

ts_cumregret = np.cumsum(ts_regret)
ab_cumregret = np.cumsum(ab_regret)
ts_pct_b = pd.Series(ts_traffic_b).rolling(200).mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(ts_cumregret, color="steelblue", label="Thompson Sampling", linewidth=2)
ax1.plot(ab_cumregret, color="tomato",    label="Fixed 50/50 A/B",   linewidth=2)
ax1.set_xlabel("Users")
ax1.set_ylabel("Cumulative regret (missed conversions)")
ax1.set_title("Cumulative regret: bandit vs fixed A/B")
ax1.legend()

ax2.plot(ts_pct_b * 100, color="steelblue", linewidth=2)
ax2.axhline(50,  color="tomato", linestyle="--", label="Fixed A/B (always 50%)")
ax2.axhline(100, color="green",  linestyle=":",  alpha=0.5, label="Perfect (always B)")
ax2.set_xlabel("Users")
ax2.set_ylabel("Traffic to arm B (%)")
ax2.set_title("Thompson Sampling adaptively shifts traffic to winning arm")
ax2.legend()

plt.tight_layout()
plt.show()

print("Total regret -- Thompson Sampling: {:.1f} missed conversions".format(ts_cumregret[-1]))
print("Total regret -- Fixed A/B:          {:.1f} missed conversions".format(ab_cumregret[-1]))
print("Regret reduction: {:.1%}".format(1 - ts_cumregret[-1] / ab_cumregret[-1]))
print("Thompson final traffic to B: {:.1%}".format(np.mean(ts_traffic_b[-1000:])))

## Summary

| Topic | Key takeaway |
|---|---|
| Confounders | Variables correlated with both treatment AND outcome corrupt causal estimates |
| Why A/B works | Randomization severs confounder → treatment link, even for unobserved confounders |
| Simple random | Unbiased but can produce unlucky imbalance with small N |
| Stratified | Guarantees balance on named dimensions at design time |
| Hash-based | Production implementation: deterministic, instant, no database needed |
| Balance check | Always verify post-assignment with SMD and Love plot |
| ANCOVA | Adjust for covariates to reduce variance or correct small post-hoc imbalances |
| Bootstrapping | Distribution-free CIs for skewed or unusual metrics |
| Thompson Sampling | Reduces user cost during experiment by adaptively shifting to winner |

Proceed to **[03 — Data Validation](03_data_validation.ipynb)**.